In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# LLM은 형식이 없는 텍스트를 생성
# LangChain에서 파서는 LLM의 출력을 원하는 형식으로 변환하는 역할

In [ ]:
# StrOutputParser
# JsonOutputParser, SimpleJsonOutputParser
# CommaSeparatedListOutputParser
# NumberedListOutputParser
# LineListOutputParser

# PydanticOutputParser
# XMLOutputParser
# YAMLOutputParser
# OutputFixingParser
# RetryWithErrorOutputParser
# PandasDataFrameOutputParser - *
# DatetimeOutputParser
# EnumOutputParser
# RegexParser

# MarkdownListOutputParser
# JsonOutputFunctionsParser - *

In [ ]:
# StrOutputParser

# LLM의 응답 객체에서 content 속성만 뽑아내어 문자열로 반환
# LLM의 응답 객체는 AIMessage 클래스 인스턴스임
# 

In [ ]:
# StrOutputParser - 1

from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# 1. 체인 구성
prompt = PromptTemplate.from_template("다음 질문에 대해 한 문장으로 대답해줘: {question}")
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
parser = StrOutputParser()

chain = prompt | model | parser
# model과 parser 사이에 LLM의 응답 객체가 있는 것

# 2. 체인 실행
result = chain.invoke({"question": "대한민국 수도는 어디야?"})

print(result)

In [4]:
# StrOutputParser - 2

from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# 1. 체인 구성
prompt = PromptTemplate.from_template("다음 질문에 대해 한 문장으로 대답해줘: {question}")
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# 파서가 없는 중간 체인
intermediate_chain = prompt | model

# 최종 체인 (파서 포함)
final_chain = intermediate_chain | StrOutputParser()

# 2. 파서가 없는 체인 실행
print("--- 1단계: 파서 없는 체인 실행 ---")
intermediate_result = intermediate_chain.invoke({"question": "대한민국 수도는 어디야?"})

# 파서가 없는 경우, 결과는 AIMessage 객체
print(f"결과 타입: {type(intermediate_result)}")
print(f"결과 내용: {intermediate_result}")
print(f"답변 텍스트: {intermediate_result.content}")
print("\n" + "="*50 + "\n")

# 3. 파서가 있는 최종 체인 실행
print("--- 2단계: 파서 있는 최종 체인 실행 ---")
final_result = final_chain.invoke({"question": "대한민국 수도는 어디야?"})

# 파서가 있는 경우, 결과는 단순한 문자열
print(f"결과 타입: {type(final_result)}")
print(f"결과 내용: {final_result}")

--- 1단계: 파서 없는 체인 실행 ---
결과 타입: <class 'langchain_core.messages.ai.AIMessage'>
결과 내용: content='대한민국의 수도는 서울입니다.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []} id='run--415e07c5-6ecb-4704-a2e7-3f9d2c16708c-0' usage_metadata={'input_tokens': 24, 'output_tokens': 10, 'total_tokens': 34, 'input_token_details': {'cache_read': 0}}
답변 텍스트: 대한민국의 수도는 서울입니다.


--- 2단계: 파서 있는 최종 체인 실행 ---
결과 타입: <class 'str'>
결과 내용: 대한민국의 수도는 서울입니다.


In [ ]:
# StrOutputParser - 3

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

from langchain_core.runnables import RunnablePassthrough

# 첫 번째 LLM 체인: 질문에 대한 답변을 생성
# 이 체인의 최종 출력은 StrOutputParser에 의해 문자열이 됩니다.
# {"question": "대한민국 수도는 어디야?"}
initial_chain = (
    PromptTemplate.from_template("질문: {question}에 대해 한 문장으로 답변해줘.")
    # "질문: 대한민국 수도는 어디야?에 대해 한 문장으로 답변해줘."
    | ChatGoogleGenerativeAI(model="gemini-2.0-flash")
    | StrOutputParser() # 답변은 문자열로 변환
)

# 두 번째 LLM 체인: 첫 번째 LLM의 답변을 요약
# 이 체인은 문자열을 입력으로 받습니다.
summary_chain = (
    PromptTemplate.from_template("다음 문장을 5단어 이내로 요약해줘: {answer}")
    | ChatGoogleGenerativeAI(model="gemini-2.0-flash")
    | StrOutputParser()
)

# 전체 체인: 첫 번째 체인과 두 번째 체인을 연결
# initial_chain의 문자열 출력이 summary_chain의 입력으로 전달

# ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
# full_chain = initial_chain | summary_chain
# {"question": "대한민국 수도는 어디야?"} + {'answer': 'initial_chain의 결과'} | summary_chain
full_chain = RunnablePassthrough.assign(answer=initial_chain) | summary_chain

# 정식대로 한다면 initial_chain의 결과값에 "answer"라는 이름표를 붙여서 다음 단계로 넘겨야 함
# summary_chain에서 {answer} 변수를 사용하기 때문

# {}는 일반 딕셔너리지만 {} |에서 {}는 RunnableParallel 객체로 변환되며
# {} 내의 체인은 병렬 실행이 됨, 이 경우 initial_chain 하나의 체인만 있는 경우

# 체인 실행
question = {"question": "대한민국 수도는 어디야?"}
result = full_chain.invoke(question)

print(f"최종 요약된 답변: {result}")

최종 요약된 답변: 서울은 대한민국의 수도.


In [ ]:
## 

In [ ]:
# RunnablePassthrough.assign(a=b): 이전 단계의 입력값을 그대로 보존하면서 새로운 키와 값을 추가
# RunnablePassthrough.assign

from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# LLM 객체
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# 입력: {"question": "대한민국 수도는 어디야?"}
initial_chain = (
    PromptTemplate.from_template("질문: {question}에 대해 한 문장으로 답변해줘.")
    # "질문: 대한민국 수도는 어디야?에 대해 한 문장으로 답변해줘."
    | llm
    | StrOutputParser()
)

# 입력: {"question": "대한민국 수도는 어디야?"}
final_assign_chain = (
    RunnablePassthrough.assign(answer=initial_chain)
    # ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
    # 1. RunnablePassthrough는 입력을 그대로 통과
    
    # 2. RunnablePassthrough.assign는 입력을 통과시키면서 initial_chain을 실행한 결과에 'answer' 키를 부여
    # 통과된 입력은 'answer' 키와 병합
    # {'question': '대한민국 수도는 어디야?', 'answer': '대한민국의 수도는 서울입니다.'}
)

# 체인 실행
question = {"question": "대한민국 수도는 어디야?"}
result = final_assign_chain.invoke(question)

print(result)
print(type(result))

{'question': '대한민국 수도는 어디야?', 'answer': '대한민국의 수도는 서울입니다.'}
<class 'dict'>


In [ ]:
# JsonOutputParser

# 1.
# LLM 응답이 JSON 형식인지 확인 → LLM은 아무리 똑똑해도 프롬프트에 따라 
# "그럴듯한" JSON을 생성하려고 시도할 뿐, 실제로 문법적으로 완벽한 JSON을 보장하지는 않음
# 2. 
# JSON 문자열 → 파이썬 dict로 변환
# 3. 
# 형식 지침 제공 (parser.get_format_instructions())

In [27]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import SimpleJsonOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. LLM 객체
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# 2. JSON 출력을 요청하는 프롬프트
prompt = PromptTemplate.from_template(
    "다음 질문에 대해 JSON 형식으로 답변해줘.\n"
    "{format_instructions}\n"
    "질문: {question}"
)

# 3. SimpleJsonOutputParser 객체
parser = SimpleJsonOutputParser()

print(parser.get_format_instructions())

# 다음 셀에서 계속

Return a JSON object.


In [24]:
# 질문 {question}과 파서 지시 {format_instructions}를 채우는 방법 - 1

In [ ]:

# 4. 체인 구성
# RunnablePassthrough를 사용해 입력과 파서 지시를 하나로
chain = (
    # 입력: {"question": "대한민국 수도는 어디야?"}
    RunnablePassthrough.assign(
        format_instructions=lambda x: parser.get_format_instructions()
    )
    # 이 시점에서 결과물은
    # {"question": "대한민국 수도는 어디야?", 'format_instructions': 'Return a JSON object.'}
    | prompt
    # "다음 질문에 대해 JSON 형식으로 답변해줘.\n"
    # "Return a JSON object.\n"
    # "질문: 대한민국 수도는 어디야?"
    | model
    | parser
)

# 5. 체인 실행
result = chain.invoke({"question": "대한민국 수도는 어디야?"})

print(result)
# 결과에 'content' 속성은 없음
# 
# 결과는 {'question': '대한민국 수도는 어디야?', 'answer': '서울'}
# 'answer' 키는 모델이 생성해서 추가
# 

print(type(result))

{'question': '대한민국 수도는 어디야?', 'answer': '서울'}
<class 'dict'>


In [ ]:
# 질문 {question}과 파서 지시 {format_instructions}를 채우는 방법 - 2

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import SimpleJsonOutputParser

# 1. LLM 설정
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# 2. JSON 형식으로 응답하도록 지시하는 프롬프트 템플릿
prompt = PromptTemplate.from_template(
    "다음 질문에 대해 JSON 형식으로 답변해줘.\n"
    "{format_instructions}\n"
    "질문: {question}"
)

# 3. JSON 파서 설정
parser = SimpleJsonOutputParser()

# 4. 체인 구성
# 입력은 {"question": "프랑스의 수도는 어디야?"}
chain = (
    RunnableLambda(lambda x: {
        "question": x["question"], # {"question": "프랑스의 수도는 어디야?"}
        "format_instructions": parser.get_format_instructions() 
        # {"format_instructions": "Return a JSON object."}

        # {"question": "프랑스의 수도는 어디야?", "format_instructions": "Return a JSON object."}
    })
    | prompt
    # "다음 질문에 대해 JSON 형식으로 답변해줘.\n"
    # "Return a JSON object.\n"
    # "질문: "프랑스의 수도는 어디야?"
    | llm
    | parser
)

# 5. 실행
result = chain.invoke({"question": "프랑스의 수도는 어디야?"})

# 6. 결과 출력
print(result)


{'question': '프랑스의 수도는 어디야?', 'answer': '파리', 'source': '지식'}


In [ ]:
# langchain의 파서에서 JsonOutputParser과 SimpleJsonOutputParser의 차이

# JsonOutputParser은 pydantic_object 지원
# Pydantic 기반으로 유효성 검사 및 타입 검증
# 이에 반해 SimpleJsonOutputParser은 단순 JSON 응답을 파싱할 때 적합

In [ ]:
# pydantic

In [ ]:
from pydantic import BaseModel

class Person(BaseModel):
# BaseModel: pydantic의 기본 클래스. 타입 검증 기능을 상속
    name: str # 문자열이어야 함
    age: int # 정수여야 함

data = {
    "name": "홍길동",
    "age": 30
}

# 자동 검증 및 객체 생성
# Person(**data): 딕셔너리를 받아서 Person 객체로 변환하면서 타입을 자동으로 체크

# person = Person(name="홍길동", age=25)
person = Person(**data)

print(person.name)  # 출력: 홍길동
print(person.age)   # 출력: 30


홍길동
30


In [ ]:
# Field

from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field(..., description="사용자의 이름")
    # name이라는 필드는 반드시 문자열 타입
    # Field(...)는 이 필드는 필수라는 의미
    # description="사용자의 이름"는 이 필드에 대한 설명
    age: int = Field(..., ge=0, le=120, description="나이는 0~120 사이여야 합니다")
    # age 필드는 정수형이어야 함
    # Field(...)는 이 필드는 필수라는 의미
    # ge=0는 값이 0 이상이어야 함; greater than or equal
    # le=120는 값이 120 이하이어야 함; less than or equal
    # description="나이는 0~120 사이여야 합니다"는 이 필드에 대한 설명

# 올바른 입력
user = User(name="홍길동", age=30)
print(user)

# 잘못된 입력
# user = User(name="홍길동", age=150)  # ValidationError 발생


In [ ]:
# partial

# 템플릿의 일부 변수는 값을 정하고
# 나머지 변수는 나중에

# 값을 언제 정하는가?에 따라 방법이 다름; partial_variables 속성을 사용하는 방법과 partial()을 호출하는 방법
# partial_variables 속성을 사용하는 방법은 템플릿을 만드는 시점에서 변수 값을 고정
# partial()을 호출하는 방법은 이미 만들어진 템플릿에 값을 동적으로 채워 넣는 방식

In [28]:
# JsonOutputParser

from langchain_core.output_parsers import JsonOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from pydantic import BaseModel

# 출력 구조 정의
class Person(BaseModel):
    name: str
    age: int

parser = JsonOutputParser(pydantic_object=Person)
# Person 클래스에 정의된 타입 유효성 검사 규칙에 따라 데이터의 유효성을 자동으로 검증

prompt = PromptTemplate(
    template="다음 정보를 JSON 형식으로 출력해 주세요: 이름은 홍길동이고 나이는 30입니다.\n{format_instructions}",
    partial_variables={"format_instructions": parser.get_format_instructions()} 
)

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

chain = prompt | model | parser

output = chain.invoke({}) # invoke()는 뭐라도 넘겨줘야 함; invoke()라고 호출할 수 없음
print(output)

{'name': '홍길동', 'age': 30}


In [29]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.runnables import RunnablePassthrough

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

class Topic(BaseModel):
    subject: str = Field(description="주제의 이름입니다.")
    category: str = Field(description="주제의 카테고리입니다.")

# 3. JsonOutputParser 객체
# Pydantic BaseModel을 사용하여 파서가 예상하는 JSON 형식을 알려줍니다.
parser = JsonOutputParser(pydantic_object=Topic)

# 4. JSON 출력을 요청하는 프롬프트
prompt = PromptTemplate.from_template(
    "다음 질문에 대해 JSON 형식으로 답변해줘.\n\n"
    "{format_instructions}\n\n"
    "질문: {question}"
)

# 5. 체인 구성
# RunnablePassthrough.assign을 사용해 입력을 병합합니다.
chain = (
    RunnablePassthrough.assign(
        question=lambda x: x['question'],
        format_instructions=lambda x: parser.get_format_instructions()
    )
    | prompt
    | model
    | parser
)

# 6. 체인 실행
result = chain.invoke({"question": "대한민국 수도는 어디야?"})

print(result)
print(type(result))

c:\Users\bayesian\miniconda3\envs\py310_64\lib\site-packages\IPython\core\interactiveshell.py:3577: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)


{'subject': '대한민국 수도', 'category': '지리'}
<class 'dict'>


In [37]:
from langchain_core.output_parsers import JsonOutputParser
from langchain.prompts import PromptTemplate
from pydantic import BaseModel, Field

class Topic(BaseModel):
    title: str = Field(..., description="주제의 제목")
    summary: str = Field(..., description="주제에 대한 간단한 요약")
    keywords: list[str] = Field(..., description="주제와 관련된 핵심 키워드 목록")

parser = JsonOutputParser(pydantic_object=Topic)

print(parser)


pydantic_object=<class '__main__.Topic'>


In [40]:

prompt = PromptTemplate(
    template="다음 주제에 대해 JSON 형식으로 정리해 주세요:\n주제: 인공지능의 미래\n{format_instructions}",
    # template="다음 주제에 대해 JSON 형식으로 정리해 주세요:\n주제: 인공지능의 미래",
    input_variables=[],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# chain = prompt | model
chain = prompt | model | parser

output = chain.invoke({}) # invoke()는 뭐라도 넘겨줘야 함; invoke()라고 호출할 수 없음
print(output)


{'title': '인공지능의 미래', 'summary': '인공지능은 사회의 다양한 측면에 걸쳐 혁신적인 변화를 가져올 것으로 예상됩니다. 자동화, 의료, 교통, 엔터테인먼트 등 다양한 분야에서 인공지능의 발전은 효율성 향상, 새로운 가능성 제시, 그리고 윤리적 및 사회적 문제 제기를 야기할 것입니다.', 'keywords': ['인공지능', '미래', '자동화', '의료', '교통', '윤리', '사회', '딥러닝', '머신러닝', 'AI']}
